# Regular expressions and our code

We've seen how regular expressions (regex) work in principle and you'll hopefully have seen already how useful they can be. A few obvious examples:

## Validation on websites

Regex is perfectly set up to validate
- Postcodes
- Dates and times
- Passwords

## Parsing program code

When someone designs a programming language they have to design both the _semantics_ (meaning) of the instructions of the language and the _syntax_ (well-formed statements). The language designer can check that statements are correctly formed using regex.

## Processing text files (eg this one)

Suppose you have the task of taking a text file written in Markdown and then rendering that into nice `html`. For instance, we have the following:

- A line of the form `# Text` is replaced by `<h1>Text<\h1>`
- A piece of text of the form `_text_` is replaced by `<emph>text<\emph>`

You get the idea... this is [how Gruber and Schwarz created Markdown](https://en.wikipedia.org/wiki/Markdown).

You can find out more by looking at Gruber's page or reading about Github flavoured markdown:
- https://daringfireball.net/projects/markdown/
- https://docs.github.com/en/get-started/writing-on-github/getting-started-with-writing-and-formatting-on-github/basic-writing-and-formatting-syntax

## In our code

Obviously there is a Python library for it. It's built in, so off we go...



In [2]:
import re

TEXT_TO_CHECK = 'Testing, Testing, 123. Today is 6 February 2025'

re.match('Testing', TEXT_TO_CHECK)

<re.Match object; span=(0, 7), match='Testing'>

This is a bit underwhelming, perhaps. But that's because we need to understand what a `re.Match` object is...

In [3]:
text_match = re.match('Testing', TEXT_TO_CHECK)

print(f'We can get the matching text: {text_match.group()}')

print(f'Starting position: {text_match.start()}, ending position: {text_match.end()}')



We can get the matching text: Testing
Starting position: 0, ending position: 7


A bit more impressed now? `match` gives us the first match and then we can unpack the results using `group()`, `start()` and `end()`.

We can find all matches if we want...

In [4]:
text_match = re.findall('Testing', TEXT_TO_CHECK)

print(f'We now find all matching text: {text_match}')


We now find all matching text: ['Testing', 'Testing']


If we know that we are going to be checking for a pattern repeatedly then it makes sense to `compile` the regex first. This makes things neat and super efficient.

In [5]:
testing_match = re.compile('Testing')

print(testing_match.findall(TEXT_TO_CHECK))

['Testing', 'Testing']


Obviously we don't really need regex to search for text - there are much less sophisticated methods. So let's do some proper pattern matching...

In [6]:
number_match = re.compile(r'[0-9]+')

all_matches = number_match.findall(TEXT_TO_CHECK)

print(all_matches)

['123', '6', '2025']


In [7]:
all_matches_as_groups = number_match.finditer(TEXT_TO_CHECK)

for match in all_matches_as_groups:
    print(f'{match.group()} starting at {match.start()} and ending at {match.end()}')


123 starting at 18 and ending at 21
6 starting at 32 and ending at 33
2025 starting at 43 and ending at 47


Remember that we can use 'anchors' to make sure we're matching at only the start or end of a pattern - this one is super useful...

In [14]:
MY_FILES = [
    'autoexec.bat',
    'compressed file.tar.gz',
    'really.bad.filename.wav',
    'not.a filename',
    '    silly spaces to ignore.txt     '
    'this is not a filename.   '
]

match_extension = re.compile(r'^\s*(.*)\.([a-z,A-Z]+)\s*$')

for filename in MY_FILES:
    match = match_extension.match(filename)
    if match:
        print(f'Filename was \'{match.group(1)}\' with extension \'{match.group(2)}\'')
    else:
        print(f'Filename \'{filename}\' is invalid')


Filename was 'autoexec' with extension 'bat'
Filename was 'compressed file.tar' with extension 'gz'
Filename was 'really.bad.filename' with extension 'wav'
Filename 'not.a filename' is invalid
Filename '    silly spaces to ignore.txt     this is not a filename.   ' is invalid


In [9]:
for filename in MY_FILES:
    match = match_extension.match(filename)
    if match:
        print(f'Full match was \'{match.group()}\'')

Full match was 'autoexec.bat'
Full match was 'compressed file.tar.gz'
Full match was 'really.bad.filename.wav'


A couple of notes from the above:
- It is worth using `r` before the text in the regex to avoid confusion with control codes like `\.`.
- We can specify the 'match groups' to return using brackets around the relevant match (eg `(.*)` and `(\w+)` above)
- Note that the `\w` symbol means a 'word character', so does the same job as `[a-z,A-Z]`
- `match.group()` (or `match.group(0)`) was the whole match rather than the first match group